# Prepare portable GEO processed data: Hippo and mixed hiPSC–K562

Run this notebook **on Sherlock**, in the environment that runs the accepted figure notebooks. It produces two real processed files from your existing data, not placeholders:

- `Hippo_timecourse_integrated.h5mu`: full broad-QC RNA object plus the frozen, aligned 50-program cNMF projection and its reference basis. Covers **Figure 3f and Figure 4c**, not the other Figure 3–4 panels from separate experiments.
- `iPSC_K562_mixed_integrated.h5mu`: QC-passing RNA, sparse cell-by-guide UMI counts and guide reference, frozen lineage/primer annotations, per-cell metrics, and embedded sequencing-saturation tables. Covers **S2i, j, l, m, n**.

The exporter does not refit Hippo cNMF. It uses the existing figure producers, writes compressed HDF5, reopens the file, reconstructs the numeric panel inputs, and checks them against pre-export references before publishing the final filename. No historical exact cell-count requirement is imposed. The original sources are never overwritten.

Run on an allocated compute/Jupyter node, not a login node: the Hippo source includes large count matrices and layers. Readback and rendering also need RAM and temporary disk space. Raw FASTQ files and the GEO study/sample metadata are separate submission components.

In [ ]:
from pathlib import Path
import os
import sys
import json
from IPython.display import display

REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "src" / "cc_perturbseq").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the manuscript-analysis repository checkout.")
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
from cc_perturbseq.geo_submission import (
    HIPPO_FILENAME, MIX_FILENAME, validate_submission, file_checksums,
)
RELEASE_DIR = Path(os.environ.get("CCPERTURBSEQ_GEO_DIR", str(REPO_ROOT / "geo_release"))).expanduser().resolve()
print("Processed-file directory:", RELEASE_DIR)


In [ ]:
from cc_perturbseq.figure_notebook import load_paths, get_path
from cc_perturbseq.geo_submission import export_hippo, export_parse_mix
from cc_perturbseq.figure_s02_parse_mix import preflight_parse_mix
import yaml

EXPORT_HIPPO = True
EXPORT_MIX = True
OVERWRITE = False  # True only when deliberately replacing a prior release.
# Optional accepted-run per-cell table. It must include ALL QC-passing cells,
# including no-guide/unassigned cells, not a single panel's filtered source table.
# In an accepted S2 session: result.cell_metrics.to_csv("Figure_S02_all_cell_metrics.csv.gz")
S2_FROZEN_METRICS = None

config_path, paths = load_paths(REPO_ROOT)
QUERY_H5AD = get_path(paths, "hippo_timecourse.query_h5ad")
PROJECTION_H5AD = get_path(paths, "hippo_timecourse.reprojected_cnmf_h5ad")
SPECTRA_TPM = get_path(paths, "hippo_rebuild.gene_spectra_tpm")
s2_config = yaml.safe_load((REPO_ROOT / "config/figure_s02_parse_mix.yaml").read_text())
S2_ROOT = Path(os.environ.get(s2_config["override_environment_variable"], s2_config["experiment_root"])).expanduser()
RELEASE_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR = RELEASE_DIR / "_temporary_validation"
TEMP_DIR.mkdir(exist_ok=True)

# Check every selected input and destination before starting expensive work.
if EXPORT_HIPPO:
    for path in (QUERY_H5AD, PROJECTION_H5AD, SPECTRA_TPM):
        if not path.is_file() or path.stat().st_size == 0:
            raise FileNotFoundError(path)
if EXPORT_MIX:
    preflight_parse_mix(S2_ROOT)
    if S2_FROZEN_METRICS is None:
        import igraph, leidenalg  # Required by the existing S2 classifier.
for enabled, filename in ((EXPORT_HIPPO, HIPPO_FILENAME), (EXPORT_MIX, MIX_FILENAME)):
    if enabled and (RELEASE_DIR / filename).exists() and not OVERWRITE:
        raise FileExistsError(f"{filename} already exists. Use the test notebook or set OVERWRITE=True deliberately.")
import anndata, mudata, scanpy
print("Input preflight passed; configuration:", config_path)
display({"Hippo RNA": str(QUERY_H5AD), "Hippo projection": str(PROJECTION_H5AD),
         "Hippo reference": str(SPECTRA_TPM), "mixed-cell source": str(S2_ROOT)})


## Export Hippo without changing its cNMF projection

In [ ]:
reports = {}
if EXPORT_HIPPO:
    reports["hippo"] = export_hippo(
        QUERY_H5AD, PROJECTION_H5AD, SPECTRA_TPM,
        RELEASE_DIR / HIPPO_FILENAME,
        overwrite=OVERWRITE, temp_parent=TEMP_DIR,
    )
    display(reports["hippo"])


## Export the mixed-cell experiment

When `S2_FROZEN_METRICS` is supplied, labels are retained from that accepted run and all guide metrics are checked against the current counts. Otherwise the existing S2 classifier runs once, using its existing parameters. Subsequent public reproduction uses the labels saved in the H5MU; it does not recluster.

In [ ]:
if EXPORT_MIX:
    reports["mixed_cells"] = export_parse_mix(
        S2_ROOT, RELEASE_DIR / MIX_FILENAME,
        frozen_metrics_csv=S2_FROZEN_METRICS, overwrite=OVERWRITE,
    )
    display(reports["mixed_cells"])


## Submission inventory

In [ ]:
manifest = {name: report["input_file"] for name, report in reports.items()}
(RELEASE_DIR / "processed_file_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
display(manifest)
print("Next run notebooks/submission/Test_GEO_Processed_Data.ipynb.")
print("Upload only the validated processed H5MU files (and desired documentation), not temporary H5ADs.")
print("FASTQ and GEO sample metadata still need their own submission entries and checksums.")
